In [1]:
import pandas as pd
import scipy.sparse as sp
import json
import anndata as ad
import scanpy as sc

In [2]:
# preview raw file
file_path= "/Users/Jan/data/exported.txt"
df = pd.read_csv(
    file_path,
    sep="\t",
    header= 0
)
df

In [ ]:
# also preview raw file
with open(file_path) as f:
    line = f.read().strip()
parts = line.split("\t")
print("Total entries:", len(parts))
print("First 20 entries:", parts[:20])

In [3]:
# save raw file as csv
df= df.transpose()
df.columns= df.iloc[0]
df= df.iloc[1:]
df.to_csv("/Users/Jan/data/test.csv", index= False) # data.csv

In [ ]:
df= pd.read_csv("/Users/Jan/data/test.csv")

In [4]:
# save metadata as a separate file
metadata= df.iloc[: 7, :]
metadata = metadata.transpose()
#metadata.to_csv("/Users/Jan/data/test_2.csv") # metadata

In [ ]:
# remove metadata from expression data
expr_df= df
expr_df= expr_df.iloc[7:, :]
expr_df.index.name= None
expr_df = expr_df.apply(pd.to_numeric, errors='coerce').fillna(0)
expr_df
#expr_df.to_csv("/Users/Jan/data/test_3.csv") # optional save

In [ ]:
# convert to anndata format

# save as sparse to save storage
adata= ad.AnnData(X= sp.csr_matrix(expr_df.T.values))

adata.var_names= expr_df.index
adata.obs_names= expr_df.columns

# fill obs with metadata
adata.obs= metadata

# sort out object types
categorical_columns= ["Sample name", "Sample Type", "Timepoint", "class"]
for col in categorical_columns:
    adata.obs[col]= adata.obs[col].astype("category")
for col in adata.obs.columns:
    if adata.obs[col].dtype== object:
        adata.obs[col]= pd.to_numeric(adata.obs[col], errors= 'coerce')

In [ ]:
# left shows the genes each cell expresses, right shows the total expression of each cell
sc.pl.violin(
    adata,
    ['Expressed genes', 'Total count'],
    jitter=0.4,
    multi_panel=True
)

In [28]:
adata.write_h5ad("/Users/Jan/data/test_4.h5ad")

In [1]:
# remove anomalies as seen in above visualisations
# copied from https://github.com/TAPE-Lab/Ramos-et-al-Trellis/blob/main/Figure_6/00_scRNA_seq_preprocessing/02_scRNA_seq_preprocessing.ipynb
# use their venv
import scanpy as sc
import matplotlib as mpl
import seaborn as sns
from matplotlib import pyplot as plt
import pandas as pd
from copy import copy
import numpy as np
import sklearn as sk
import scprep
import phate
#import scrublet
import random
import anndata as ad
import tqdm as tqdm
import scipy.sparse as sp

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.set_figure_params(dpi=100, color_map = "viridis", frameon=True, transparent=True,
                    dpi_save=800, facecolor="None", format="pdf", figsize=[4,4])

# Figure output directory
sc.settings.figdir = "C:/Users/Jan/data/chris outputs"

# Set seed for reproducibility
np.random.seed(12)

In [ ]:
adata= sc.read_h5ad("/Users/Jan/data/test_4.h5ad")
adata.var['mt'] = adata.var_names.str.startswith('mt-')
adata.var["mt"]

In [4]:
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
umi_total_log10 = np.log10(adata.obs.total_counts)
umi_med = umi_total_log10.median()
adata.obs['log10_total_counts'] = umi_total_log10
Q1 = umi_total_log10.quantile(0.25)
Q3 = umi_total_log10.quantile(0.75)
IQR = Q3 - Q1
IQR

In [ ]:
umi_cutoff_up = umi_med + (2 * IQR)

print("median log10(UMI): ", umi_med)
print("Upper log10(UMI): ", umi_cutoff_up)

In [ ]:
adata.obs['outlier_mt'] = adata.obs.pct_counts_mt > 20
adata.obs['outlier_total'] = adata.obs.log10_total_counts > umi_cutoff_up
adata.obs['lowq_total'] = adata.obs.total_counts < 1000
print('%u cells with high %% of mitochondrial genes' % (sum(adata.obs['outlier_mt'])))
print('%u cells with large total counts' % (sum(adata.obs['outlier_total'])))
print('%u cells with low number of total counts' % (sum(adata.obs['lowq_total'])))
adata= adata[~adata.obs['outlier_mt'], :]
adata= adata[~adata.obs['outlier_total'], :]
adata= adata[~adata.obs['lowq_total'], :]

In [ ]:
adata.var_names_make_unique()
sc.pp.filter_cells(adata, min_genes=400)
sc.pp.filter_genes(adata, min_cells=25)

In [ ]:
sc.pl.violin(
    adata,
    ['Expressed genes', 'Total count'],
    jitter=0.4,
    multi_panel=True
)

In [ ]:
adata.obs_names_make_unique()
sc.external.pp.scrublet(adata, expected_doublet_rate=0.03, threshold=0.19)

In [11]:
adata.layers['counts']=adata.X.copy()

In [20]:
sc.pp.normalize_total(adata, target_sum=1e4, exclude_highly_expressed=True, max_fraction=0.05)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes( 
    adata,
    n_top_genes=5000,
    flavor="seurat_v3",
    layer="counts",
    subset=False
)

In [22]:
adata.raw= adata

In [23]:
sc.pp.scale(adata, max_value=10) # Scale each gene to unit variance. Clip values exceeding standard deviation 10.
sc.tl.pca(adata, n_comps=100, svd_solver='arpack')

In [24]:
adata.obs['predicted_doublet'] = adata.obs['predicted_doublet'].astype(str).astype('category')

In [ ]:
sc.pl.pca(adata, color=['Sample name', 'doublet_score', 'predicted_doublet'], components=['1,2'], ncols=1)

In [ ]:
sc.external.tl.phate(adata, t=7, random_state=12)

In [ ]:
sc.external.pl.phate(adata, color=['Sample name', 'doublet_score', 'predicted_doublet'], ncols=1,  
frameon=False, add_outline=True, title='')

In [ ]:
sc.pp.neighbors(adata, n_pcs=50, n_neighbors=100, random_state=12)
sc.tl.leiden(adata, resolution = 0.1, random_state=12, key_added="leidenr0.1")
sc.external.pl.phate(adata, color='leidenr0.1', use_raw=True, 
frameon=False, add_outline=True, title='')

In [ ]:
# this cell removes doublets. it was originally followed by another doublet removal bit of code, but I'll skip it
# since it looks highly specific to the original dataset
### this code might not remove doublets as expected. if this is the case, run the below cell instead
adata.obs['doublet_outliers'] = adata.obs.predicted_doublet.isin(['True'])
print('%u cells with high probability of being doublets' % (sum(adata.obs['doublet_outliers'])))
adata = adata[~adata.obs['doublet_outliers'], :]

In [ ]:
adata = adata[~adata.obs['predicted_doublet'].astype(bool), :]

In [ ]:
sc.external.pl.phate(adata, color=['Sample name'], ncols=1,  
frameon=False, add_outline=True, title='')

In [ ]:
sc.external.tl.phate(adata, t=7, random_state=12)

In [ ]:
sc.tl.pca(adata, n_comps=100, svd_solver='arpack')

# High level clustering to identfy cell types
sc.pp.neighbors(adata, random_state=12, n_neighbors=100)
sc.tl.leiden(adata, resolution = 0.1, random_state=12, key_added="leidenr0.1")

In [36]:
adata.write_h5ad(filename = "C:/Users/Jan/data/preprocessed_data.h5ad")

In [21]:
# cell type inference
# switch back to the other env
import scanpy as sc
import numpy as np
import seaborn as sns
import celltypist
import matplotlib.pyplot as plt
import pandas as pd
from celltypist import models
from scipy.stats import median_abs_deviation, pearsonr
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

In [3]:
# consult R file to convert Chris' dataset to h5ad
adata= sc.read_h5ad("C:/Users/Jan/data/INTepi_CLEAN.h5ad")

In [ ]:
print(adata.obs.columns.tolist())

In [4]:
# filtering so that only WT and AK cells are used for training
adata.var_names = adata.var["features"].astype(str)
adata.var_names_make_unique()
conditions= ["monoWT", "monoAK"]
adata= adata[adata.obs["orig.ident"].isin(conditions)]

In [5]:
# sort out object types
for col in adata.obs.columns:
    adata.obs[col] = pd.to_numeric(
        adata.obs[col],
        errors="ignore"
    )
for col in adata.obs.columns:
    if adata.obs[col].dtype == "object":
        adata.obs[col] = adata.obs[col].astype("category")

In [ ]:
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from matplotlib import pyplot as plt
from scipy.sparse import issparse
sc.settings.verbosity = 0
sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    # color_map="YlGnBu",
    frameon=False,
)

rcb.logger.setLevel(logging.ERROR)
ro.pandas2ri.activate()
%load_ext rpy2.ipython

In [ ]:
# plotting the expression of each cell
p1 = sns.histplot(adata.obs["nCount_RNA"], bins=100, kde=False)

In [7]:
# log1p transform
scales_counts = sc.pp.normalize_total(adata, target_sum=None, inplace=False)
adata.layers["log1p_norm"] = sc.pp.log1p(scales_counts["X"], copy=True)

In [8]:
adata_prediction= sc.read_h5ad("C:/Users/Jan/data/preprocessed_data.h5ad")

In [ ]:
# model testing
train_adata, test_adata = train_test_split(adata, test_size=0.25, random_state=12, stratify=adata.obs['curatedCLUST'])
model = celltypist.train(train_adata, labels = 'curatedCLUST', n_jobs = 10, feature_selection = True)
model.write("C:/Users/Jan/data/model_test.pkl")

In [ ]:
# still testing
predictions = celltypist.annotate(test_adata, model = model, majority_voting = True)

In [ ]:
# more testing
#results = predictions.to_adata()
#results= results.obs[["curatedCLUST", "predicted_labels", "majority_voting"]]
true= results["curatedCLUST"].astype(str)
preds= results["predicted_labels"].astype(str)
mv_preds= results["majority_voting"].astype(str)
labels = sorted(set(true) | set(preds))
accuracy = (true == preds).mean()
print(f"Model Accuracy: {accuracy * 100:.2f}%")
accuracy = (true == mv_preds).mean()
print(f"Majority voting Accuracy: {accuracy * 100:.2f}%")
cm = confusion_matrix(true, preds, labels=labels, normalize= "true")
fig, ax = plt.subplots(figsize=(10, 10))
ax.matshow(cm, cmap=plt.cm.Blues, alpha=0.3)
ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='left')
ax.set_yticklabels(labels)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(x=j, y=i, s=f"{cm[i, j]:.2f}", 
                va='center', ha='center', size='large')

plt.xlabel('Predicted Labels', fontsize=14)
plt.ylabel('True Labels', fontsize=14)
plt.title('CellTypist Confusion Matrix', fontsize=16, pad=20)
plt.tight_layout()
plt.savefig("C:/Users/Jan/data/chris outputs/integrated_final/full dataset/cell types/proportion confusion matrix")
plt.show()

In [ ]:
cm = confusion_matrix(true, preds, labels=labels)
fig, ax = plt.subplots(figsize=(10, 10))
ax.matshow(cm, cmap=plt.cm.Blues, alpha=0.3)
ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='left')
ax.set_yticklabels(labels)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(x=j, y=i, s=cm[i, j], 
                va='center', ha='center', size='large')

plt.xlabel('Predicted Labels', fontsize=14)
plt.ylabel('True Labels', fontsize=14)
plt.title('CellTypist Confusion Matrix', fontsize=16, pad=20)
plt.tight_layout()
plt.savefig("C:/Users/Jan/data/chris outputs/integrated_final/full dataset/cell types/count confusion matrix")
plt.show()

In [ ]:
# retraining with full dataset
model = celltypist.train(adata, labels = 'curatedCLUST', n_jobs = 10, feature_selection = True)

In [ ]:
model.write("C:/Users/Jan/data/model_2.pkl")

In [9]:
model= models.Model.load("C:/Users/Jan/data/model_2.pkl")

In [ ]:
# normalise only if needed
#sc.pp.normalize_total(adata_prediction, target_sum= 1e4)
#sc.pp.log1p(adata_prediction)
predictions = celltypist.annotate(adata_prediction, model = model, majority_voting = True)

In [ ]:
adata_results = predictions.to_adata()
sc.tl.umap(adata_results)

In [ ]:
sc.settings.figdir = "C:/Users/Jan/data/chris outputs"
sc.settings.set_figure_params(dpi=300)
sc.pl.umap(adata_results, color= "predicted_labels")#, save= "_cell_types_updated.png")

In [ ]:
sc.pl.umap(adata_results, color= "Sample name", save= "_condition_umap_updated.png")

In [28]:
adata_results.write_h5ad("/Users/Jan/data/final_data.h5ad")

In [2]:
adata_results= sc.read_h5ad("/Users/Jan/data/final_data.h5ad")

In [ ]:
# harmony integration
# use trellis env
adata_results.obs["predicted_labels"]= (adata_results.obs["predicted_labels"].replace({"Goblet / DCS": "Goblet or DCS", 'Outlier 1 (Stressed?)': 'Outlier 1 (Stressed)'}))

In [ ]:
sc.pp.pca(adata_results)

In [ ]:
pcs= adata.obsm["X_pca"]

In [ ]:
harmony_out= hm.run_harmony(pcs, adata_results.obs, "Sample Type")

In [ ]:
adata_results.obsm['X_pca_harmony'] = harmony_out.Z_corr

In [ ]:
sc.pp.neighbors(adata_results, use_rep='X_pca_harmony')

In [ ]:
sc.tl.umap(adata_results)

In [ ]:
sc.tl.leiden(adata_results)

In [ ]:
del adata_results.obsm["X_phate"]
ph= phate.PHATE(random_state= 12)
adata_results.obsm["X_phate"]= ph.fit_transform(adata_results.obsm["X_pca_harmony"])

In [10]:
# drop cells classified as outliers because there are only six of them and they're skewing graphs
adata_results= adata_results[adata_results.obs["predicted_labels"]!= "Outlier 1 (Stressed)"]

In [32]:
plt.rcdefaults()
sns.reset_defaults()